In [1]:
import requests

# Sprawdź czy API z Ćw. 2 nadal działa
try:
    r = requests.get('http://localhost:8001/health', timeout=2)
    print('API działa:', r.json())
except Exception as e:
    print('API niedostępne:', e)

API działa: {'status': 'ok', 'model': 'RandomForestClassifier'}


In [3]:
# Pytania kontrolne:

# 1. Jakie cechy ma model z Ćw. 2?
#    ODPOWIEDŹ: 4 cechy — amount, hour, is_electronics, tx_per_day

# 2. Co zwraca endpoint POST /score?
#    ODPOWIEDŹ: JSON z polami is_fraud (True/False) i fraud_probability (0-1)

# 3. Dlaczego w ml_consumer.py używamy tx_per_minute=5 (stała)?
#    ODPOWIEDŹ: Producent nie wysyła tej cechy w transakcjach, więc używamy
#    wartości domyślnej. W systemie produkcyjnym byłaby wyliczana z historii
#    transakcji użytkownika (np. liczba tx w ostatniej minucie).

# 4. Co by się stało gdyby dwa procesy ml_consumer.py miały ten sam group_id?
#    ODPOWIEDŹ: Kafka rozdzieliłaby partycje między konsumentów (load balancing)
#    — każda transakcja byłaby przetwarzana tylko przez jednego konsumenta,
#    nie przez obu. To mechanizm skalowania w Kafce.

print("Pytania kontrolne — done")

Pytania kontrolne — done


In [4]:
# Problem: Random Forest to uczenie NADZOROWANE
# Potrzebuje etykiet: które transakcje są fraudami?
#
# W rzeczywistości:
# - Etykiety zbieramy tygodniami/miesiącami
# - Nowe rodzaje fraudów nie mają etykiet
# - Frauderzy adaptują swoje zachowanie
#
# Rozwiązanie: uczenie NIENADZOROWANE
# Trenujemy model tylko na NORMALNYCH transakcjach.
# Każda transakcja mocno odbiegająca od normy = podejrzana.

print("Uczenie nadzorowane (RF):")
print("  Dane treningowe: 2000 normalnych + 100 fraudów")
print("  Model uczy się: 'to jest fraud, to nie jest'")
print()
print("Uczenie nienadzorowane (IF):")
print("  Dane treningowe: 2000 normalnych (fraudy niepotrzebne!)")
print("  Model uczy się: 'jak wygląda normalna transakcja'")
print("  Na żywo: 'ta transakcja NIE wygląda normalnie'")

Uczenie nadzorowane (RF):
  Dane treningowe: 2000 normalnych + 100 fraudów
  Model uczy się: 'to jest fraud, to nie jest'

Uczenie nienadzorowane (IF):
  Dane treningowe: 2000 normalnych (fraudy niepotrzebne!)
  Model uczy się: 'jak wygląda normalna transakcja'
  Na żywo: 'ta transakcja NIE wygląda normalnie'


In [5]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Generujemy TYLKO normalne transakcje — fraudów nie potrzebujemy!
N_NORMAL = 2000

normal = pd.DataFrame({
    'amount':         np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute':  np.random.poisson(3, N_NORMAL),
})

print(f"Dane treningowe: {len(normal)} normalnych transakcji")
print()
normal.describe().round(2)

Dane treningowe: 2000 normalnych transakcji



,amount,is_electronics,tx_per_minute
count,2000.00,2000.00,2000.00
mean,253.77,0.28,2.96
std,324.84,0.45,1.65
min,5.81,0.00,0.00
25%,79.63,0.00,2.00
50%,155.20,0.00,3.00
75%,293.82,1.00,4.00
max,5000.00,1.00,10.00


In [6]:
from sklearn.ensemble import IsolationForest
import pickle

features = ['amount', 'is_electronics', 'tx_per_minute']
X_train = normal[features]

# contamination: spodziewamy się ~5% anomalii w strumieniu
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)
iso_forest.fit(X_train)

print("Model wytrenowany.")
print(f"Liczba drzew: {iso_forest.n_estimators}")
print(f"Contamination: {iso_forest.contamination}")

# Zapisz model
with open('fraud_model_if.pkl', 'wb') as f:
    pickle.dump(iso_forest, f)
print("\nZapisano do fraud_model_if.pkl")

Model wytrenowany.
Liczba drzew: 100
Contamination: 0.05

Zapisano do fraud_model_if.pkl


In [7]:
test_cases = pd.DataFrame([
    {'amount': 150.0,  'is_electronics': 0, 'tx_per_minute': 3,  'opis': 'normalna'},
    {'amount': 89.0,   'is_electronics': 0, 'tx_per_minute': 2,  'opis': 'normalna'},
    {'amount': 4800.0, 'is_electronics': 1, 'tx_per_minute': 12, 'opis': 'podejrzana'},
    {'amount': 3500.0, 'is_electronics': 1, 'tx_per_minute': 9,  'opis': 'podejrzana'},
    {'amount': 250.0,  'is_electronics': 1, 'tx_per_minute': 5,  'opis': 'graniczna'},
])

X_test = test_cases[features]
preds  = iso_forest.predict(X_test)           # +1 lub -1
scores = iso_forest.decision_function(X_test) # anomaly score

test_cases['wynik']    = preds
test_cases['score']    = scores.round(4)
test_cases['anomalia'] = test_cases['wynik'] == -1

print(test_cases[['opis', 'amount', 'is_electronics', 'tx_per_minute',
                  'wynik', 'score', 'anomalia']].to_string(index=False))

      opis  amount  is_electronics  tx_per_minute  wynik   score  anomalia
  normalna   150.0               0              3      1  0.2115     False
  normalna    89.0               0              2      1  0.2097     False
podejrzana  4800.0               1             12     -1 -0.1932      True
podejrzana  3500.0               1              9     -1 -0.1894      True
 graniczna   250.0               1              5      1  0.0641     False


In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Naprawa: RF z Ćw. 2 oczekuje 4 cech (amount, hour, is_electronics, tx_per_day)
# Dodajemy brakujące cechy do danych testowych

features_rf = ['amount', 'hour', 'is_electronics', 'tx_per_day']
features_if = ['amount', 'is_electronics', 'tx_per_minute']

# Dodaj brakujące cechy do test_df
np.random.seed(1)
# Normalne: dzień (8-22), niskie tx_per_day
test_df.loc[test_df['true_label'] == 0, 'hour'] = np.random.randint(8, 22, 50)
test_df.loc[test_df['true_label'] == 0, 'tx_per_day'] = np.random.poisson(3, 50)
# Fraudy: noc (0-5), wysokie tx_per_day
test_df.loc[test_df['true_label'] == 1, 'hour'] = np.random.randint(0, 5, 10)
test_df.loc[test_df['true_label'] == 1, 'tx_per_day'] = np.random.poisson(8, 10)

# Predykcje — RF używa 4 cech, IF używa 3
rf_pred = rf_model.predict(test_df[features_rf])                        # 0 lub 1
if_pred = (iso_forest.predict(test_df[features_if]) == -1).astype(int)  # 0 lub 1

y_true = test_df['true_label']

print(f"{'Model':<20} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 55)
for name, pred in [("Random Forest", rf_pred), ("Isolation Forest", if_pred)]:
    p = precision_score(y_true, pred, zero_division=0)
    r = recall_score(y_true, pred, zero_division=0)
    f = f1_score(y_true, pred, zero_division=0)
    print(f"{name:<20} {p:>10.3f} {r:>10.3f} {f:>10.3f}")

print()
print("Uwaga: RF widział fraudy podczas treningu, IF nie — dlatego RF zwykle wypada lepiej")
print("na danych syntetycznych. W produkcji (bez etykiet) IF jest jedyną opcją.")

Model                 Precision     Recall         F1
-------------------------------------------------------
Random Forest             1.000      1.000      1.000
Isolation Forest          0.625      1.000      0.769

Uwaga: RF widział fraudy podczas treningu, IF nie — dlatego RF zwykle wypada lepiej
na danych syntetycznych. W produkcji (bez etykiet) IF jest jedyną opcją.


In [14]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np

app = FastAPI(title="Fraud Detection API — Isolation Forest")

model = pickle.load(open('fraud_model_if.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.post("/score")
def score(tx: Transaction):
    X = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    prediction    = model.predict(X)[0]            # +1 lub -1
    anomaly_score = model.decision_function(X)[0]  # ujemny = bardziej podejrzany

    # Normalizujemy score do przedziału [0, 1] — dla spójności z Ćw. 2
    fraud_probability = float(np.clip(0.5 - anomaly_score, 0.0, 1.0))

    return {
        "is_fraud":          bool(prediction == -1),
        "fraud_probability": round(fraud_probability, 4),
        "model":             "isolation_forest",
    }

@app.get("/health")
def health():
    return {"status": "ok"}

Overwriting fraud_api.py


In [15]:
import requests, time

time.sleep(1)  # daj chwilę na restart serwera

cases = [
    {"amount": 150,  "is_electronics": 0, "tx_per_minute": 3,  "opis": "normalna"},
    {"amount": 4800, "is_electronics": 1, "tx_per_minute": 12, "opis": "podejrzana"},
    {"amount": 89,   "is_electronics": 0, "tx_per_minute": 2,  "opis": "normalna"},
    {"amount": 3200, "is_electronics": 1, "tx_per_minute": 8,  "opis": "podejrzana"},
]

for case in cases:
    payload = {k: v for k, v in case.items() if k != 'opis'}
    r = requests.post("http://localhost:8001/score", json=payload)
    result = r.json()
    print(f"[{case['opis']:10s}] amount={case['amount']:5} "
          f"→ fraud={result['is_fraud']}, prob={result['fraud_probability']:.3f}")

[normalna  ] amount=  150 → fraud=False, prob=0.288
[podejrzana] amount= 4800 → fraud=True, prob=0.693
[normalna  ] amount=   89 → fraud=False, prob=0.290
[podejrzana] amount= 3200 → fraud=True, prob=0.678


In [16]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json, requests

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='ml-scoring-if',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

API_URL = "http://localhost:8001/score"

for message in consumer:
    tx = message.value
    
    # cechy zgodne z API IF (3 cechy)
    features = {
        "amount": tx['amount'],
        "is_electronics": 1 if tx.get('category') == 'elektronika' else 0,
        "tx_per_minute": 5
    }
    
    # Odpytaj API
    response = requests.post(API_URL, json=features)
    result = response.json()
    
    # Jeśli fraud — wyślij alert
    if result.get('is_fraud'):
        alert = {**tx, **result}
        alert_producer.send('alerts', value=alert)
        print(f"🚨 ML ALERT | {tx['tx_id']} | {tx['amount']:.2f} PLN | prob={result['fraud_probability']}")

Overwriting ml_consumer.py


In [17]:
# Pytania dyskusyjne — Zadanie 5.2

# 1. Jakie transakcje IF flaguje, a RF nie (i odwrotnie)?
#    OBSERWACJA: IF flaguje transakcje które ODBIEGAJĄ od normalnych wzorców
#    (wysoka kwota + elektronika + dużo tx/min). RF flaguje to czego się nauczył
#    z etykietowanych fraudów. IF może wykryć NOWE typy fraudów których RF nie widział,
#    ale RF jest precyzyjniejszy dla znanych wzorców.

# 2. Czy parametr contamination=0.05 ma wpływ na liczbę alertów?
#    Co by się stało gdybyś zmienił go na 0.01?
#    ODPOWIEDŹ: Tak, contamination definiuje "próg" anomalii. Przy 0.05 model
#    spodziewa się ~5% anomalii. Przy 0.01 model byłby bardziej restrykcyjny —
#    flagowałby tylko ~1% najbardziej odstających transakcji (mniej alertów,
#    ale wyższa precyzja). Przy 0.10 — więcej alertów, ale więcej fałszywych alarmów.

# 3. Jaką zaletę ma IF w systemie produkcyjnym gdzie fraudy są nowe i nieznane?
#    ODPOWIEDŹ: IF nie potrzebuje etykiet — uczy się tylko na normalnych transakcjach.
#    To pozwala wykrywać NOWE typy fraudów których nigdy wcześniej nie widzieliśmy.
#    W produkcji etykiety zbiera się tygodniami/miesiącami, a frauderzy ciągle 
#    zmieniają taktyki. IF jest pierwszą linią obrony — flaguje "wszystko co dziwne",
#    a potem człowiek lub RF weryfikuje.

print("Pytania dyskusyjne — done")

Pytania dyskusyjne — done


In [18]:
# === ZADANIE DOMOWE 1: Eksperyment z contamination ===

# Wytrenuj 3 modele z różnymi wartościami contamination
contaminations = [0.01, 0.05, 0.10]
models = {}

for c in contaminations:
    m = IsolationForest(n_estimators=100, contamination=c, random_state=42)
    m.fit(X_train)
    models[c] = m

# Sprawdź ile transakcji flaguje każdy model na tych samych danych testowych
print(f"{'Contamination':>15} {'Liczba alertów':>20} {'% z 60 transakcji':>20}")
print("-" * 60)
for c, m in models.items():
    preds = m.predict(test_df[features_if])
    n_alerts = (preds == -1).sum()
    pct = n_alerts / len(test_df) * 100
    print(f"{c:>15.2f} {n_alerts:>20} {pct:>19.1f}%")

print()
print("WNIOSEK:")
print("- contamination=0.01 → mniej alertów, wyższa precyzja, mogą umknąć fraudy")
print("- contamination=0.05 → balans (nasze ustawienie)")
print("- contamination=0.10 → więcej alertów, więcej fałszywych alarmów")
print()
print("W praktyce contamination ustawia się na podstawie historycznego odsetka fraudów.")

  Contamination       Liczba alertów    % z 60 transakcji
------------------------------------------------------------
           0.01                   10                16.7%
           0.05                   16                26.7%
           0.10                   20                33.3%

WNIOSEK:
- contamination=0.01 → mniej alertów, wyższa precyzja, mogą umknąć fraudy
- contamination=0.05 → balans (nasze ustawienie)
- contamination=0.10 → więcej alertów, więcej fałszywych alarmów

W praktyce contamination ustawia się na podstawie historycznego odsetka fraudów.


In [19]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np

app = FastAPI(title="Fraud Detection API — Isolation Forest")

model = pickle.load(open('fraud_model_if.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.post("/score")
def score(tx: Transaction):
    X = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    prediction    = model.predict(X)[0]
    anomaly_score = model.decision_function(X)[0]
    fraud_probability = float(np.clip(0.5 - anomaly_score, 0.0, 1.0))
    return {
        "is_fraud":          bool(prediction == -1),
        "fraud_probability": round(fraud_probability, 4),
        "model":             "isolation_forest",
    }

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/model-info")
def model_info():
    return {
        "type":          "isolation_forest",
        "n_estimators":  model.n_estimators,
        "contamination": model.contamination,
    }

Overwriting fraud_api.py


In [20]:
# Test nowego endpointu /model-info
r = requests.get("http://localhost:8001/model-info")
print(r.json())

{'type': 'isolation_forest', 'n_estimators': 100, 'contamination': 0.05}


In [21]:
%%file fraud_api_rf.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np

app = FastAPI(title="Fraud Detection API — Random Forest")

model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    hour: int
    is_electronics: int
    tx_per_day: int

@app.post("/score")
def score(tx: Transaction):
    X = np.array([[tx.amount, tx.hour, tx.is_electronics, tx.tx_per_day]])
    is_fraud = bool(model.predict(X)[0])
    fraud_probability = float(model.predict_proba(X)[0][1])
    return {
        "is_fraud":          is_fraud,
        "fraud_probability": round(fraud_probability, 4),
        "model":             "random_forest",
    }

@app.get("/health")
def health():
    return {"status": "ok"}

Writing fraud_api_rf.py


In [22]:
%%file ml_consumer_rf.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json, requests

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='ml-scoring-rf-compare',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

API_URL = "http://localhost:8002/score"

for message in consumer:
    tx = message.value
    
    hour = int(tx.get('hour', datetime.fromisoformat(tx['timestamp']).hour))
    features = {
        "amount": tx['amount'],
        "hour": hour,
        "is_electronics": 1 if tx.get('category') == 'elektronika' else 0,
        "tx_per_day": 5
    }
    
    response = requests.post(API_URL, json=features)
    result = response.json()
    
    if result.get('is_fraud'):
        alert = {**tx, **result}
        alert_producer.send('alerts', value=alert)
        print(f"🌲 RF ALERT | {tx['tx_id']} | {tx['amount']:.2f} PLN | prob={result['fraud_probability']}")

Writing ml_consumer_rf.py


In [23]:
git config --global user.name "Maria Norek"
git config --global user.email "mn148956@student.sgh.waw.pl"

SyntaxError: invalid syntax (1673004875.py, line 1)